In [7]:
import sys
import os

target_dir = os.path.abspath('..') 

if target_dir not in sys.path:
    sys.path.append(target_dir)

In [8]:
# Use helpers
from preprocess import * 
from understatapi import UnderstatClient

from preprocess.player_stats import get_position_players_stats_df

understat = UnderstatClient()

# Fit model to forwards for proof of concept
stats = ["goals", "xG", "assists", "xA", "key_passes", "xGChain", "xGBuildup"]
window_size = 10

f_stats = get_position_players_stats_df(understat, ['F'], stats)

In [9]:
# train/test split
train_len = int(len(f_stats) * 0.8)
train_df = f_stats.iloc[:train_len]
test_df = f_stats.iloc[train_len:]

In [10]:
from preprocess.player_stats import *

# create datasets
train_dataset = CustomFootballDataset(train_df, window_size, multiple_players=False)
test_dataset = CustomFootballDataset(test_df, window_size, multiple_players=False)

In [11]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False
)

In [13]:
from football_lstm import *
import torch.optim as optim

model = FootballLSTM(n_features=len(f_stats.columns), hidden_size=32)
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 20

model.train_model(
    optimizer=optimizer,
    loss_fn=loss_fn,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    n_epochs=num_epochs
)

Train total loss: 3085.7310170829296
Test total loss: 1381.7414801120758
Train total loss: 3058.156762212515
Test total loss: 1379.6459710747004
Train total loss: 3054.4295399188995
Test total loss: 1383.9216523319483
Train total loss: 3051.344670832157
Test total loss: 1376.4967097342014
Train total loss: 3049.4917126595974
Test total loss: 1379.3182771652937
Train total loss: 3047.7239125967026
Test total loss: 1377.459991261363
Train total loss: 3044.501780539751
Test total loss: 1376.7154966294765
Train total loss: 3046.3776319026947
Test total loss: 1378.102767020464
Train total loss: 3042.404438138008
Test total loss: 1376.288651138544
Train total loss: 3057.083160698414
Test total loss: 1376.8869827389717
Train total loss: 3038.295059502125
Test total loss: 1374.7525081336498
Train total loss: 3037.3495911955833
Test total loss: 1375.7573564946651
Train total loss: 3033.9056536257267
Test total loss: 1375.5676048994064
Train total loss: 3034.2721025645733
Test total loss: 1376.4